# Implement the Gymnasium Environment

In this notebook, you will implement the core methods of a [Gymnasium](https://gymnasium.farama.org/) environment for the battery scheduling problem.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import gymnasium as gym
from envs.battery_env import BatteryStorageEnv

## The Gymnasium Interface

Gymnasium provides a standard interface for RL environments. You can find the docs for their Env class [here](https://gymnasium.farama.org/api/env/). A simple example can be found [here](https://gymnasium.farama.org/introduction/create_custom_env/).

### Core Methods

Every Gymnasium environment implements two key methods:

| Method | Returns | Description |
|--------|---------|-------------|
| `reset()` | `(observation, info)` | Start a new episode |
| `step(action)` | `(observation, reward, terminated, truncated, info)` | Take one action and advance the environment |

<br>

> **Note:** Gymnasium also defines `render()` and `close()` for environments with visual output (game screens, animations). Since our battery environment has no visual rendering, we don't need these.

### Key Attributes

| Attribute | Description |
|-----------|-------------|
| `action_space` | Defines valid actions. All valid actions should be contained within the space. More on spaces [here](https://gymnasium.farama.org/api/spaces/#gymnasium.spaces.Space). |
| `observation_space` | Defines valid observations. All valid observations should be contained within the space. |
| `np_random` | Random number generator for the environment. Should be set in the first line of the `reset()` function by this line: `super().reset(seed=seed)`. |

**Spaces examples:**
- Discrete action space (e.g. `0` and `1`): `action_space = gym.spaces.Discrete(2)`
- Continuous action space (e.g. in between -1 and 1): `action_space = spaces.Box(low=-1.0, high=1.0, shape=(1,), dtype=np.float32)`

The agent-environment loop looks like this:

```python
obs, info = env.reset()

while not terminated and not truncated:
    action = agent.choose_action(obs)
    obs, reward, terminated, truncated, info = env.step(action)

env.close()
```

| Return value | Description |
|-------------|-------------|
| `observation` | What the agent sees (numpy array, normalized to [0, 1]) |
| `reward` | Scalar feedback signal (higher = better) |
| `terminated` | `True` when the episode ends naturally |
| `truncated` | `True` if cut short externally (we don't use this) |
| `info` | Debug dictionary with human-readable state |

To give you an idea, we demonstrate a typical agent-environment loop in the next cell using the [CartPole-v1](https://gymnasium.farama.org/environments/classic_control/cart_pole/) environment of Gymnasium:

In [2]:
# Create the environment. 
env = gym.make("CartPole-v1") # We use the default render_mode ("None") to avoid PyGame issues on MacOS. You can add render_mode="human" if you want to see the animation (but be aware of the closing issue for MacOS).
# You always need to reset the env before the first step to initialize it and get the initial observation.
obs, info = env.reset(seed=42)
# To understand the outputs, please visit the documentation of the CartPole-v1 environment: https://gymnasium.farama.org/environments/classic_control/cart_pole/
print("Initial observation:", obs)
print("Observation space:", env.observation_space)
print("Action space:", env.action_space)

total_reward = 0
terminated = False
truncated = False
step = 0

while not terminated and not truncated:
    action = env.action_space.sample()  # random action (0 = push left, 1 = push right)
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    step += 1


print(f"\nEpisode finished after {step} steps with total reward: {total_reward}")
print(f"Last observation: {obs}")

# Always remember to close the environment when done (especially if it has a render mode that opens a window)
# Note that on MacOS PyGame is buggy and cannot be closed properly, so we use render_mode="None" to avoid that issue.
env.close()

Initial observation: [ 0.0273956  -0.00611216  0.03585979  0.0197368 ]
Observation space: Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)
Action space: Discrete(2)

Episode finished after 16 steps with total reward: 16.0
Last observation: [ 0.20497012  0.78401965 -0.22891165 -1.3960086 ]


## Your Task

Open the file **`01_workshop/envs/battery_env.py`** and implement the 4 methods below the line:

```python
# =========================================================================
# METHODS FOR PARTICIPANTS TO IMPLEMENT
# =========================================================================
```

Everything above that line is already implemented for you (data loading, action/observation spaces, forecasting, degradation model).

### Implementation Order

| # | Method | What it does |
|---|--------|-------------|
| 1 | `_calculate_reward()` | determines the reward given to the agent |
| 2 | `_get_obs()` | Build the observation array |
| 3 | `reset()` | Initialize a new episode | 
| 4 | `step()` | Process one action and update state | 

After implementing each method, come back to this notebook and run the corresponding test cell.

## Method 1: `_calculate_reward(load, charge_power, price)`

The reward tells the RL agent how good its action was. The method returns a float number.

- **Charging** (`charge_power > 0`): We buy extra electricity → higher grid cost
- **Discharging** (`charge_power < 0`): Battery offsets load → lower grid cost
- **Can't sell to grid**: If discharge exceeds load, the excess is wasted → clamp `grid_energy` to 0

**Go implement `_calculate_reward()` now, then run the cell below.**

In [3]:
# Reload the module to pick up your latest changes from battery_env.py
# (Jupyter caches imports, so without this you'd still run the old version. Alternatively you could restart the kernel, but this is more convenient for quick iterations.)
import importlib
import envs.battery_env as _mod
importlib.reload(_mod)
from envs.battery_env import BatteryStorageEnv

# Create an instance of your BatteryStorageEnv to test the reward function
env = BatteryStorageEnv()

# Test 1: Charging — grid supplies load (1.0) + charge (1.5) = 2.5 kWh
reward = env._calculate_reward(load=1.0, charge_power=1.5, price=0.20)
assert reward == -0.5, f"Expected -0.5, got {reward}"
print(f"Test 1 passed: Charging reward = {reward}")

# Test 2: Discharging — grid supplies load (1.0) - discharge (0.5) = 0.5 kWh
reward = env._calculate_reward(load=1.0, charge_power=-0.5, price=0.20)
assert reward == -0.1, f"Expected -0.1, got {reward}"
print(f"Test 2 passed: Discharging reward = {reward}")

# Test 3: Over-discharge — grid_energy would be negative, but clamped to 0
reward = env._calculate_reward(load=0.5, charge_power=-2.0, price=0.20)
assert reward == 0.0, f"Expected 0.0, got {reward}"
print(f"Test 3 passed: Over-discharge reward = {reward}")

print("\nAll tests passed!")

NotImplementedError: Implement this method

## Method 2: `_get_obs()`

The observation is a **normalized array** (all values in [0, 1]) that tells the RL agent about the current state. It should contain:

| Index | Value | How to compute |
|-------|-------|---------------|
| 0 | State of charge | `soc / max_capacity` |
| 1 | Battery health | `self.health` (already in [0, 1]) |
| 2 | Hour of day | `hours_of_day[current_step] / 24.0` |
| 3, 4 | Current price, load | `self._get_forecast(0)` |
| 5, 6 | Next hour price, load | `self._get_forecast(1)` |
| ... | ... | ... |

Use `self._get_forecast(h)` which returns a `(price, load)` tuple, already normalized.

**Important:** Normalize SoC with `self.max_capacity` (not `self.capacity`), so the value stays in [0, 1] even when capacity degrades.

**Go implement `_get_obs()` now, then move on to `reset()`.**

In [ ]:
# Reload the module to pick up your latest changes from battery_env.py
import importlib
import envs.battery_env as _mod
importlib.reload(_mod)
from envs.battery_env import BatteryStorageEnv

env = BatteryStorageEnv()

# Manually set up environment state (since reset() isn't implemented yet)
env._current_prices = env.prices[0]
env._current_loads = env.loads[0]
env._current_hours_of_day = env.hours_of_day[0]
env._current_days_of_week = env.days_of_week[0]
env.current_step = 0
env.health = 1.0

# --- Test 1: Correct shape ---
env.soc = 5.0
obs = env._get_obs()
expected_dim = 5 + 2 * env.forecast_horizon
assert obs.shape == (expected_dim,), f"Expected shape ({expected_dim},), got {obs.shape}"
print(f"Test 1 passed: Observation shape is {obs.shape}")

# --- Test 2: All values in [0, 1] ---
assert np.all(obs >= 0.0) and np.all(obs <= 1.0), f"Values out of bounds: min={obs.min()}, max={obs.max()}"
print(f"Test 2 passed: All values in [0, 1] (min={obs.min():.3f}, max={obs.max():.3f})")

# --- Test 3: SoC normalized by max_capacity, not capacity ---
# Simulate degraded battery: capacity dropped but max_capacity stays the same
env.soc = 5.0
env.capacity = 7.0  # degraded
env.max_capacity = 10.0  # original
obs = env._get_obs()
expected_soc_norm = 5.0 / 10.0  # should use max_capacity
assert obs[0] == expected_soc_norm, f"SoC should be {expected_soc_norm} (using max_capacity), got {obs[0]}. Did you normalize by capacity instead of max_capacity?"
print(f"Test 3 passed: SoC correctly normalized by max_capacity (5.0/10.0 = {obs[0]})")

# Reset capacity back to normal
env.capacity = env.max_capacity

print("\nAll tests passed!")

Test 1 passed: Observation shape is (13,)
Test 2 passed: All values in [0, 1] (min=0.000, max=1.000)
Test 3 passed: SoC correctly normalized by max_capacity (5.0/10.0 = 0.5)

All tests passed!


## Method 3: `reset()`

This method initializes a new episode. The steps are:

1. **`super().reset(seed=seed)`** — Must be called first! Sets up `self.np_random` for reproducible randomness.
2. **Select a random episode** from `self._available_episodes` (a `(start, end)` tuple of indices).
3. **Load episode data** into `self._current_prices`, `self._current_loads`, `self._current_hours_of_day`, `self._current_days_of_week` from the chunked arrays (e.g. `self.prices[episode_idx]`).
4. **Reset capacity** to `self.max_capacity` (in case it degraded during the previous episode).
5. **Random initial SoC** in `[0, capacity]`.
6. **Reset** `self.current_step = 0` and `self.health = 1.0`.
7. **Return** `(self._get_obs(), self._get_info())`.

Use `self.np_random` for all randomness:
- `self.np_random.integers(low, high)` — random int in [low, high)
- `self.np_random.uniform(low, high)` — random float in [low, high)

**Go implement `reset()` now, then run the test cell below.**

In [ ]:
# Reload the module to pick up your latest changes from battery_env.py
import importlib
import envs.battery_env as _mod
importlib.reload(_mod)
from envs.battery_env import BatteryStorageEnv

# --- Test 1: reset() returns a tuple of (observation, info) ---
env = BatteryStorageEnv()
result = env.reset(seed=42)
assert isinstance(result, tuple) and len(result) == 2, f"reset() should return (obs, info) tuple, got {type(result)}"
obs, info = result
print("Test 1 passed: reset() returns (obs, info) tuple")

# --- Test 2: Observation has correct shape and bounds ---
expected_dim = 5 + 2 * env.forecast_horizon
assert obs.shape == (expected_dim,), f"Expected obs shape ({expected_dim},), got {obs.shape}"
assert np.all(obs >= 0.0) and np.all(obs <= 1.0), f"Obs values out of [0,1]: min={obs.min()}, max={obs.max()}"
print(f"Test 2 passed: Observation shape {obs.shape}, all values in [0, 1]")

# --- Test 3: State is properly initialized ---
assert env.current_step == 0, f"current_step should be 0 after reset, got {env.current_step}"
assert 0 <= env.soc <= env.capacity, f"SoC should be in [0, {env.capacity}], got {env.soc}"
assert env.health == 1.0, f"Health should be 1.0 after reset, got {env.health}"
assert env.capacity == env.max_capacity, f"Capacity should be reset to max_capacity ({env.max_capacity}), got {env.capacity}"
print(f"Test 3 passed: current_step=0, soc={env.soc:.2f}, health=1.0, capacity={env.capacity}")

# --- Test 4: Info dict contains expected keys ---
expected_keys = {"soc", "step", "episode_idx", "price", "load", "hour_of_day", "day_of_week", "health", "capacity"}
assert expected_keys.issubset(info.keys()), f"Info dict missing keys: {expected_keys - info.keys()}"
print(f"Test 4 passed: Info dict contains all expected keys")

# --- Test 5: Same seed produces same results (reproducibility) ---
obs1, info1 = env.reset(seed=123)
obs2, info2 = env.reset(seed=123)
assert np.array_equal(obs1, obs2), "Same seed should produce identical observations"
assert info1["soc"] == info2["soc"], "Same seed should produce identical SoC"
print("Test 5 passed: Same seed produces identical results")

# --- Test 6: Different seeds produce different episodes ---
obs_a, info_a = env.reset(seed=1)
obs_b, info_b = env.reset(seed=999)
# At least SoC or episode index should differ
assert info_a["soc"] != info_b["soc"] or info_a["episode_idx"] != info_b["episode_idx"], \
    "Different seeds should (very likely) produce different episodes"
print("Test 6 passed: Different seeds produce different episodes")

print("\nAll tests passed!")

Test 1 passed: reset() returns (obs, info) tuple
Test 2 passed: Observation shape (13,), all values in [0, 1]
Test 3 passed: current_step=0, soc=4.39, health=1.0, capacity=10.0
Test 4 passed: Info dict contains all expected keys
Test 5 passed: Same seed produces identical results
Test 6 passed: Different seeds produce different episodes

All tests passed!


You should see **6 tests passed**. These tests verify that:
- `reset()` returns a valid `(observation, info)` tuple
- Observation has the correct shape and values in [0, 1]
- State is properly initialized (SoC, step counter, health, capacity)
- Info dict contains all expected keys
- Seeding produces reproducible results
- Different seeds select different episodes

## Method 4: `step()`

This is the main simulation method. Each call processes one hour:

1. **Extract action** from the action array: `action = action[0]`
2. **Compute charge power**: `charge_power = action * self.max_charge_rate`
3. **Clip new SoC** to `[0, capacity]`: `new_soc = np.clip(self.soc + charge_power, 0, self.capacity)`
4. **Effective power** (accounts for battery limits): `charge_power_effective = new_soc - self.soc`
5. **Get current price and load** from `self._current_prices[self.current_step]`
6. **Calculate reward** using `self._calculate_reward(load, charge_power_effective, price)`
7. **Update state**: `self.soc = new_soc`, increment `self.current_step`
8. **Check termination**: `terminated = (self.current_step >= self.episode_length)`
9. **Degradation** (Level 2): If `self.enable_degradation`, call `self._apply_degradation(charge_power_effective)` and subtract `self.health_weight * health_damage` from the reward.
10. **Return** `(self._get_obs(), reward, terminated, False, self._get_info())`

**Go implement `step()` now, then run the test cell below.**

In [ ]:
# Reload and run ALL tests
import importlib
import envs.battery_env as _mod
importlib.reload(_mod)

!uv run pytest ../../tests/test_battery_env.py -v 

============================= test session starts ==============================
platform darwin -- Python 3.13.7, pytest-9.0.2, pluggy-1.6.0 -- /Users/david.goll/Documents/projects/workshop-rl2-implementation/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/david.goll/Documents/projects/workshop-rl2-implementation
configfile: pyproject.toml
plugins: anyio-4.12.1
collected 21 items                                                             

../../tests/test_battery_env.py::TestBatteryEnv::test_gymnasium_api_compliance PASSED [  4%]
../../tests/test_battery_env.py::TestBatteryEnv::test_observation_space_shape PASSED [  9%]
../../tests/test_battery_env.py::TestBatteryEnv::test_action_space_shape PASSED [ 14%]
../../tests/test_battery_env.py::TestBatteryEnv::test_reset_returns_valid_observation PASSED [ 19%]
../../tests/test_battery_env.py::TestBatteryEnv::test_reset_initializes_state PASSED [ 23%]
../../tests/test_battery_env.py::TestBatteryEnv::test_reset_seeding_reproducibili

**All 21 tests should pass.** If some tests fail, check the error messages — they usually point to exactly what's wrong.

## Sanity Check

Let's run your environment for a few steps to see it in action.

In [ ]:
import importlib
import envs.battery_env as _mod
importlib.reload(_mod)
from envs.battery_env import BatteryStorageEnv

env = BatteryStorageEnv()
obs, info = env.reset(seed=42)

print(f"Observation shape: {obs.shape}")
print(f"Initial SoC:    {info['soc']:.2f} kWh")
print(f"Initial health: {info['health']:.2f}")
print()

for i in range(5):
    action = env.action_space.sample() # random action
    obs, reward, terminated, truncated, info = env.step(action)
    print(f"Step {i+1}: action={action[0]:+.2f}  reward={reward:.4f}  soc={info['soc']:.2f}  price={info['price']:.3f}")

env.close()

Observation shape: (13,)
Initial SoC:    4.39 kWh
Initial health: 1.00

Step 1: action=-0.24  reward=0.0000  soc=3.79  price=0.095
Step 2: action=+0.19  reward=-0.0785  soc=4.26  price=0.116
Step 3: action=+0.59  reward=-0.2209  soc=5.73  price=0.096
Step 4: action=-0.98  reward=0.0000  soc=3.28  price=0.129
Step 5: action=-0.02  reward=-0.0735  soc=3.23  price=0.114


## Next Steps

Your environment is ready! In the next notebook, we will use it to:
- Compare baseline policies (heuristic, MPC, LP)
- Train a PPO agent using Stable-Baselines3
- Evaluate how RL compares to classical approaches